# Agent Workshop

An agent here is a folder: a written personality, some skills, and some evals.
This notebook publishes one, talks to it, scores it, changes it, and scores it
again.

This is the purpose-built agent pattern from the talk: one author's domain
knowledge, packaged so other people can use it. The personality controls how
the agent behaves, the skills define what it can do, and the evals measure
whether a change helped.
[Shippy](https://allenai.org/blog/shippy-deep-dive) is built the same way.

The example reports recent earthquakes near a place. Change its personality and
skills if you want it to do something else, or leave it alone and just run the
loop.

Setup, from a terminal (File > New > Terminal):

```bash
cd ~/agent-workshop
git pull --ff-only
pip install -e cli
```

The JupyterHub image was built from an older copy of this repo, so pull first.
The pull updates `workshop.ipynb` itself: after it finishes, close this notebook
and reopen it, so you are reading the current version rather than the one the
image shipped with.

## 1. Setup

In [ ]:
%run setup_ai2_workshop_demo.py

In [ ]:
NAME = ""   # <-- your name, lowercase, no spaces

import os

AGENT_DIR = "quake-watch"           # the folder under agents/
SLUG      = f"{NAME}-quake-watch"   # what it is called on the platform

assert NAME, "put your name above"
# Every command below acts as you: your agent, your sandbox, your threads.
os.environ["MOTHERSHIP_EXTERNAL_ID"] = NAME

print("publishing as", SLUG)

Everything you create is named after you, so nothing collides with anyone else in the room.

In [ ]:
!mothership agents search --limit 5

A table back, even an empty one, means you can reach the platform.

## 2. The agent folder

Open `agents/quake-watch/` in the file browser. The whole agent is there:

- `SOUL.md` — the personality, written as a system prompt. It scopes the agent
  to one job (earthquakes near a place, in a time window, from the live USGS
  catalog, never from memory) and sets rules for answering: depth alongside
  every magnitude, times in local and UTC, state the search that was run. Read
  the last section: a general chatbot will happily guess when the next
  earthquake is, and this one is told not to. Step 7 checks it obeys.
- `skills/` — one folder per capability, each a markdown file the agent reads
  when it needs it. There are two: `geocode` turns a place name into
  coordinates, and `usgs-quakes` queries the USGS earthquake catalog.
- `evals/` — the tests, one file each: does it report real recent activity,
  does it refuse to predict, does it name which Springfield it picked.
  `mothership evals run` sends them to the platform, which puts the question
  to your agent and grades what comes back.
- `agent.json` — the config: name, harness, model, and two parameters (search
  radius, magnitude floor) that reach the agent as environment variables. The
  personality and skills are the agent; this file is how one deployment of it
  is set up.

The `usgs-quakes` skill follows the same policy as Shippy: the agent never
assembles raw API calls. The skill tells it to run a script, and the script
owns the HTTP call, the parameters, and the output format. When the agent uses
the skill, it runs this:

In [ ]:
!python3 agents/{AGENT_DIR}/skills/usgs-quakes/scripts/quakes.py \
    --latitude 61.218 --longitude -149.900 --days 3

## 3. Publish

One command, four steps you could run by hand:

1. Package the agent folder into a tarball.
2. Upload it to the workshop bucket as this version's frozen copy.
3. Register the agent under your slug. On a re-run, add a new version instead
   and make it the current one.
4. Stop any running copy of the agent, so the next conversation starts on the
   new version.

Every version points at one shared runtime image; when your agent starts, that
image downloads your tarball and boots from it. Publishing takes seconds.

In [ ]:
!mothership publish {AGENT_DIR} --slug {SLUG}

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> publish the quake-watch agent

</details>

The platform assigned your agent an id (the `agent_id` line above). Every
later command needs it, so look it up from your slug rather than pasting it;
that way it can never disagree with the name you set in section 1.

In [ ]:
import json, subprocess

result = subprocess.run(
    ["mothership", "--json", "agents", "search", "--slug.eq", SLUG],
    capture_output=True, text=True)
agents = next((v for v in json.loads(result.stdout).values() if isinstance(v, list)), [])
assert agents, f"{SLUG} is not in the catalog; run the publish cell above"
AGENT_ID = agents[0]["agent_id"]
print("agent_id", AGENT_ID)

## 4. Start a sandbox

The agent version you published is a definition, not a running process. This
step provisions the machine it runs on: a sandbox, an isolated environment
that downloads your agent version and boots it.

The sandbox is keyed by `external_id`. Here that is your username, so you get
a sandbox that is yours alone. In a real deployment you choose what the key
means, and the choice sets the concurrency model. Key by Slack channel and
many people share one agent and its state. Key by project name and one person
gets a separate sandbox per project, each with its own state. Any
infrastructure that sandboxes agents has to make this decision somewhere; in
Mothership it is this key.

In production this step is invisible. Sending a message creates the sandbox if
one is not running, and the user just sees a slower first reply; the CLI
merges it the same way (`mothership messages submit` "ensures a sandbox is
running", per its own help text). Here it is separate so the lifecycle is
visible.

In [ ]:
!mothership sandboxes create {AGENT_ID}

In [ ]:
import json, subprocess, time

for _ in range(60):  # up to 5 minutes
    result = subprocess.run(
        ["mothership", "--json", "sandboxes", "search",
         "--agent-id.eq", AGENT_ID, "--external-id.eq", NAME],
        capture_output=True, text=True)
    rows = next((v for v in json.loads(result.stdout).values() if isinstance(v, list)), [])
    state = rows[0]["state"] if rows else "no sandbox yet"
    if state == "running":
        print(f"running: sandbox_id {rows[0]['sandbox_id']}")
        break
    print(state, end="  ", flush=True)
    time.sleep(5)
else:
    print("\nnot running after 5 minutes -- flag an instructor")

A sandbox moves through `created`, `starting`, `running`, `stopping`,
`stopped`. Yours is `running` now; section 10 walks it down the other half.

## 5. Send messages

Your sandbox is already running, so even the first reply comes back without a
cold start.

In [ ]:
!mothership messages submit "Any notable earthquakes near Anchorage this week?" \
    --agent-id {AGENT_ID} --timeout 600

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> ask my agent about earthquakes near Anchorage

</details>

A thread is the conversation. The first submit created one (the `Thread ...
created` line in its output); passing its id back continues that conversation,
so the agent sees the earlier exchange. Paste the id below, then ask a
follow-up the agent was told not to answer.

In [ ]:
THREAD_ID = ""   # <-- paste the thread id printed above (Thread thr_... created)

assert THREAD_ID, "paste the thread_id from the first message"

In [ ]:
!mothership messages submit "Does that mean a bigger one is coming?" \
    --agent-id {AGENT_ID} --thread-id {THREAD_ID}

### The browser chat

The chat UI at the link below is built on the same APIs the CLI has been
calling: the same agent, the same sandbox, the same thread. Nothing about your
agent is CLI-specific, so any application can be a client of it; this UI is
one demo of that. In the URL, `agent` is what you published, `external_id` is
the name you set in section 1, and the path is the thread id you pasted above,
so it opens the conversation mid-stream.

Stopping the sandbox in section 10 does not end this browser session. The next
message does the invisible restart from section 4: a new sandbox comes up, the
first reply is slower, and the conversation picks up where it left off.

In [ ]:
CHAT_URL = f"https://agents.proto.skylight.earth/chat/{THREAD_ID}?agent={AGENT_ID}&external_id={NAME}"
print(CHAT_URL)

## 6. What is an agent eval

An agent is not deterministic. The same question can come back different on
different runs, and a personality edit that fixes one behavior can quietly
break another. Running the agent and reading a few answers does not tell you
whether it got better; it tells you whether the runs you happened to read
looked fine. An eval is the alternative: a task the agent is measured against,
repeatably.

A task has two parts: a prompt to put to the agent, and criteria for grading
what comes back. A criterion is a named rubric with a weight; an LLM judge
reads the agent's answer against each rubric and scores it. Criteria are
written to be assigned to tasks, not buried in them: a rule like "never
forecast" belongs on every task this agent has, because no earthquake answer
is allowed to predict, whatever the question was.

Two properties make the scores worth trusting. Evals run on the same
infrastructure real conversations use, your published agent in a sandbox, so
the eval measures the thing you ship rather than a copy of it. And results
attach to the agent version that produced them, so a score is a fact about a
specific published version, not about the agent in general.

Once the set of evals is comprehensive, larger changes become measurable: swap
the harness or the LLM and see what it did to every score, or rework how a
skill is implemented and check the agent still behaves, instead of hoping it
does.

The evals here were already authored; they are the three files in `evals/`.
The next cell registers them against your agent, and the one after lists what
the platform now holds: each task, and the criteria assigned to it. The loop
you are about to run, score a baseline, change one thing, score again, is the
whole method.

In [ ]:
!mothership evals sync {AGENT_DIR} --slug {SLUG}

In [ ]:
import json, subprocess

query = json.dumps({"agent_id": {"eq": AGENT_ID}})
result = subprocess.run(
    ["mothership", "evals", "search", "--resource", "tasks", "--query", query],
    capture_output=True, text=True)
for task in json.loads(result.stdout)["records"]:
    judge = next(s for s in task["spec"]["scorers"] if s["kind"] == "llm_judge")
    print(task["slug"])
    for criterion in judge["criteria"]:
        print("   ", criterion["name"])

## 7. Run the evals

The task is `recent-activity`. It asks the same Anchorage question you asked
in section 5 and grades the answer on five criteria:

- `used_the_live_catalog` — the answer shows a real query: specific events, or
  an explicit empty result. Remembered facts about Alaska score zero.
- `reported_depth_with_magnitude` — every event carries a depth alongside its
  magnitude.
- `times_are_readable_and_correct` — timestamps are readable, in Alaska local
  time and UTC, not raw epoch numbers.
- `stated_the_search_it_ran` — the answer says what was searched: the center,
  the radius, the magnitude floor, the time window.
- `no_unprompted_forecast` — nothing forward-looking anywhere in the answer.

Scores run 0 to 1, one per criterion and one for the task overall. 0.8 and up
is a pass, under 0.5 is a fail, and in between is needs-work. The
per-criterion lines in the report carry the names above, and they say which
one to go fix.

In [ ]:
!mothership evals run {AGENT_DIR} --slug {SLUG} --task recent-activity

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> run the evals for my agent

</details>

Copy the `run_id` from the bottom of that report, because you will compare
against it in a moment.

### What the test actually is

The platform is holding your three tasks now, and it will hand them back as
[Harbor](https://www.harborframework.com/docs) task directories. The next cell
asks for them, and unpacks the zip that comes back into `harbor-tasks/`, one
directory per task.

In [ ]:
import json
BODY = json.dumps({"agent_id": AGENT_ID})

!mothership evals export --out eval-tasks.zip --body '{BODY}'
!rm -rf harbor-tasks && unzip -qo eval-tasks.zip -d harbor-tasks && find harbor-tasks -type f | sort

Six files per task, and between them they are the whole test. In the
`recent-activity` directory:

- `instruction.md` — the question the agent is asked.
- `tests/judge.toml` — one `[[criterion]]` block per criterion, each with the
  rubric it grades against, its points and its weight. The names are the ones
  you just read down the left of the report.
- `tests/rubric.md` — the reference the judge is handed: what is true about
  Anchorage, and what a good answer has to do.
- `tests/test.sh` — what the verifier runs, which is rewardkit.
- `task.toml` — name, tags, timeouts.
- `environment/Dockerfile` — the image the agent answers from, whichever one
  you published.

Mothership pins Harbor (`harbor==0.16.1`, `harbor-rewardkit==0.1.7`) and treats
it as the authority on this layout: its own tests load these rendered files back
through stock Harbor's parsers, so a Harbor release that changed the format
would fail there rather than here. The grading is not trapped in the platform.
Anything that reads a Harbor task directory reads these.

In [ ]:
!cat harbor-tasks/{SLUG}-recent-activity/task.toml
!head -40 harbor-tasks/{SLUG}-recent-activity/tests/judge.toml

`judge.toml` is where the score you just read comes from. Edit a rubric there
and you have changed what counts as good; edit `SOUL.md` and you have changed
the agent's chances of meeting it. The next section does the second one.

## 8. Change the agent, rerun the evals

Open `agents/quake-watch/SOUL.md` and change one thing, aimed at whichever line
of the report scored lowest. For example:

- Low on `reported_depth_with_magnitude`: say depth is required on every
  earthquake you mention, not just encouraged.
- Low on `stated_the_search_it_ran`: say every answer must state the radius,
  the smallest magnitude, and the time window you searched.
- Low on `times_are_readable_and_correct`: say every time must be given in both
  local time and UTC.

Change one thing only. Change two and you will not know which one worked.

In [ ]:
diff = !git diff --stat agents/{AGENT_DIR}/SOUL.md
print("\n".join(diff) if diff else "SOUL.md is unchanged. Edit it, then re-run this cell.")

Publish the change (seconds, same command), then run the same test against it.

In [ ]:
!mothership publish {AGENT_DIR} --slug {SLUG}

In [ ]:
BASELINE = ""   # <-- paste the run_id from section 7

assert BASELINE, "paste the earlier run_id"

In [ ]:
!mothership evals run {AGENT_DIR} --slug {SLUG} --task recent-activity --previous {BASELINE}

The last columns show what moved. Anything under about 0.1 is noise, because
the agent and the grader both vary between runs. If nothing moved, that is a
real answer too: the change you were sure about did nothing.

## 9. Call the agent from Claude Code

The CLI and the chat UI are two clients of your agent. A general-purpose agent
can be a third. The mechanism is the same one your agent uses internally: a
skill that wraps a CLI. `usgs-quakes` tells your agent how to run `quakes.py`;
the skill the next cell writes tells Claude Code how to run
`mothership messages submit` against your agent, and tells it to defer to your
agent on anything seismic rather than answering from its own memory.

Claude Code discovers skills under `.claude/skills/` in whatever repo it is
opened in, so the file goes there, with your slug, agent id, and name filled
in.

In [ ]:
from pathlib import Path

SKILL = Path(".claude/skills/ask-quake-watch/SKILL.md")
SKILL.parent.mkdir(parents=True, exist_ok=True)
SKILL.write_text(f'''---
name: ask-quake-watch
description: "Ask the {SLUG} agent about earthquakes near a place. Use for any question about seismic activity, recent quakes, magnitudes, or whether something was an earthquake. Do not answer these from memory; this agent queries the live USGS catalog."
---

# Ask {SLUG}

Run from the repo root. Pass the user's question through as written.

    MOTHERSHIP_EXTERNAL_ID={NAME} mothership messages submit "<question>" \\
        --agent-id {AGENT_ID} --timeout 600

The output includes a `Thread thrd_... created` line the first time. To keep a
conversation going, pass that id back on later questions with `--thread-id`,
so the agent sees the earlier exchange.

The first reply can take a minute if the agent's sandbox has to start. Relay
the agent's answer as it stands, including the search it says it ran and
anything it declines to answer. Do not add a forecast.
''')
print(SKILL.read_text())

Open a terminal, `cd ~/agent-workshop`, run `claude`, and ask something the
skill was not written for, like "any earthquakes near Tokyo this week?".
Claude Code reads the skill, calls your agent, and relays the answer. The
purpose-built agent did the work; the general-purpose one found it and
delegated.

This needs Claude Code installed and signed in. If it is not on this machine,
the file you just wrote works from any checkout of this repo that has the
`mothership` CLI configured.

## 10. Stop the sandbox

Stop it so it is not left running.

In [ ]:
!mothership sandboxes stop --agent-id {AGENT_ID}

## 11. What to build next

1. **Your domain instead of earthquakes.** Swap `usgs-quakes` for a script
   over an API you already run: sensor telemetry, inventory, a weather feed,
   your product's own search. The shape of quake-watch (resolve the question
   to parameters, run the query, format the answer under rules) fits most
   lookup problems as they stand. The evals will need replacing too, since
   they stop measuring anything once the agent is not about earthquakes.
2. **An agent inside your own application.** The browser chat in section 5
   was a separate app calling the agent over HTTP; a panel inside your product
   is the same call from a different page. Users ask questions where they
   already are, and the agent answers with your data and your rules.
3. **A team Slack bot.** Point the agent at a channel and it answers there:
   what an alert means, who owns a service, where the runbook is, what shipped
   this week. The skills wrap whatever your team already uses to answer those
   questions by hand.
4. **A set of team Claude Code skills.** Section 9 in reverse: instead of one
   skill that calls one agent, a folder of skills that wrap your team's
   internal tools, checked into the repo so everyone's Claude Code can query
   the metrics database, look up a customer, or file a ticket in the house
   format.
5. **A first pass on a queue.** Support tickets, incoming PRs, alerts,
   applications. The agent reads each one, classifies it, and drafts a
   response for a person to approve. The evals grade the drafts, which is how
   you find out whether it is good enough to trust with the easy ones.
6. **An institutional-memory agent.** Skills over your docs, your wiki, and
   your decision records, with a persona that says "I don't know" when the
   answer isn't written down. The eval is the set of questions a new hire asks
   in their first month.

Whatever the use case, it is the same folder: a persona, some skills, and the
evals that say whether it works. Nothing in it is specific to the platform it
ran on today.